# graft: gold-patch calibration

Verifies every training task's reference patch in the notebook's subprocess sandbox, so dev scores can be
read against what is solvable here. Writes `/kaggle/working/gold_calibration.jsonl`. CPU only.

## 1. Install the harness wheelhouse

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in ('LiteLLM', 'LiteLLM Router', 'LiteLLM Proxy', 'litellm', 'httpx', 'httpcore',
                 'openai', 'urllib3', 'opentelemetry', 'opentelemetry.context'):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Verify every gold patch

In [ ]:
import asyncio
import concurrent.futures
import json
import time
from adk_submission import ModelRegistry
from swegemma.config import EvalConfig
from swegemma.deduplication import resolve_task_snapshot_paths
from swegemma.evaluate import Evaluator
from swegemma.harness.verification import verify_task
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
OUT = WORKING_DIR / 'gold_calibration.jsonl'

config = EvalConfig(
    tasks_path=DATA_DIR / 'tasks.jsonl',
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'calibration',
    submission_dir=DATA_DIR / 'sample_submission',
    models=ModelRegistry(),
    sandbox='subprocess',
    graph_dir=str(DATA_DIR / 'graphs'),
    embeddings_dir=str(DATA_DIR / 'embeddings'),
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)
evaluator = Evaluator(config)


def run_sync(coro):
    """Run a coroutine to completion even though the notebook kernel already has an event loop."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()

tasks = load_tasks(config.tasks_path)

done = set()
if OUT.exists():
    done = {json.loads(l)['id'] for l in OUT.read_text().splitlines() if l.strip()}

for idx, task in enumerate(tasks, start=1):
    if task.instance_id in done:
        continue
    t0 = time.time()
    row = {'id': task.instance_id, 'repo': task.repo}
    try:
        snapshot_path, base_snapshot_path, patch_path = resolve_task_snapshot_paths(
            config.snapshots_dir, task.instance_id, task.repo)
        res = run_sync(verify_task(
            evaluator.docker, config, task, snapshot_path,
            base_snapshot_path=base_snapshot_path, patch_path=patch_path,
            agent_patch=task.patch, start_time=time.perf_counter()))
        row.update(gold_resolved=bool(res.resolved), test_exit_code=res.test_exit_code,
                   error=str(getattr(res, 'error', '') or '')[:300],
                   test_output_tail=(res.test_output or '')[-1500:])
    except Exception as exc:
        row.update(gold_resolved=False, error=f'{type(exc).__name__}: {exc}'[:300])
    row['seconds'] = round(time.time() - t0, 1)
    with OUT.open('a') as f:
        f.write(json.dumps(row) + '\n')
    print(f'[{idx}/{len(tasks)}] {task.instance_id}: gold_resolved={row["gold_resolved"]} '
          f'({row["seconds"]}s){" " + row["error"][:120] if row.get("error") else ""}', flush=True)

## 3. Summary

In [ ]:
import pandas as pd
df = pd.read_json(OUT, lines=True)
print(f'gold resolves {df.gold_resolved.sum()}/{len(df)} tasks in this sandbox')
display(df.groupby('repo').gold_resolved.agg(['sum', 'count']))
display(df[~df.gold_resolved][['id', 'test_exit_code', 'error']])